In this notebook, we generate the precipitation variance timeseries, used in Eq. 8 of the paper.

### *Matérn Correlation Matrix generation* 

**Goal**: is to generate the common correlation structure $R$ jointly from three precipitation data sets.To that end, we fix the Matérn  parameters in the correlations function (length scale "$l_s$", smoothness "$k$").

In [1]:
import numpy as np
import pandas as pd
import os
from scipy.spatial.distance import pdist, squareform
from scipy.special import kv, gamma
import glob
import rioxarray
import warnings
warnings.filterwarnings('ignore')

- define helper functions

In [2]:
def get_coords_vals(raster_path):
    "func to extract coordinates and values from raster data"
    with rioxarray.open_rasterio(raster_path) as rds:
        data = rds.squeeze().values
        x_coords, y_coords = np.meshgrid(rds.x.values, rds.y.values)
        coords = np.column_stack((x_coords.ravel(), y_coords.ravel()))
        mask = ~np.isnan(data.ravel())
        return coords[mask], data.ravel()[mask]

In [3]:
def matern_correlation(distance, phi, kappa):
    scaled_distance = np.maximum(distance / max(phi,1e-8), 1e-8)  # avoid division by 0
    cor = 1 / (2**(kappa-1) * gamma(kappa)) * (scaled_distance**kappa) * kv(kappa, scaled_distance)
    return np.maximum(cor, 0)  # +ve

In [4]:
def create_R_matern(phi, kappa, distances, R=None):
    RR = matern_correlation(distances, phi, kappa)
    if R is None:
        R = squareform(RR)
    else:
        R[:] = squareform(RR)
    np.fill_diagonal(R, 1.0)
    return R

- generate the common correlation matrix using fixed parameters

In [5]:
ls_fixed = 50  # km
k_fixed = 1.5

template = '../data/p/trmm/trmm_2003.01.01.tif'   
coords, _ = get_coords_vals(template)
distances = pdist(coords, metric='euclidean') * 111  # km

# R
R = create_R_matern(ls_fixed, k_fixed, distances)
#save
#pd.DataFrame(R).to_csv('../output/R_matrix.csv', index=False)

- read data from rasters

In [ ]:
pmax_folder = 'data/Pmax'
pmin_folder = 'data/Pmin'

pmax_files = sorted(glob.glob(os.path.join(pmax_folder, '*.tif')))
pin_files = sorted(glob.glob(os.path.join(pmin_folder, '*.tif')))

common_dates = [os.path.basename(f)[-14:-4] for f in pmax_files]

In [ ]:
s_means = {}
v_means = {}

for date_str in tqdm(common_dates):
    pmax_file = os.path.join(pmax_folder, f'Pmax_{date_str}.tif')
    pmin_file = os.path.join(pmin_folder, f'Pmin_{date_str}.tif')
    
    with rioxarray.open_rasterio(pmax_file) as rds:
        emax_data = rds.squeeze().values.astype(np.float32)
    with rioxarray.open_rasterio(pmin_file) as rds:
        emin_data = rds.squeeze().values.astype(np.float32)
    
    pmax_vals = emax_data.flatten()[valid_mask]
    pmin_vals = emin_data.flatten()[valid_mask]
    
    # diagonal S (0.5 * |Emax - Emin|)
    s_diag = 0.25 * np.abs(pmax_vals - pmin_vals)
    s_means[date_str] = np.mean(s_diag)
    
    # Efficient computation of the mean of V = S @ R @ S
    # For a matrix product S @ R @ S, where S is diagonal,
    # the mean can be computed as: mean(S @ R @ S) = (s @ R @ s) / n²
    # where s is the vector of diagonal elements of S and n is the dimension
    
    # thus we ue here the vector form instead of the diagonal matrix
    s_vector = s_diag
  
    Rs = R @ s_vector  
    sRs = s_vector @ Rs  #cov

    n = len(s_vector)
    v_mean = sRs / (n * n)
    v_means[date_str] = v_mean
    
    print(f"Date: {date_str}, S mean: {s_means[date_str]:.2f}, V mean: {v_mean:.2f}")

# Save mean values
pd.DataFrame.from_dict(s_means, orient='index', columns=['Smean']).to_csv('../output/SPmean.csv')
pd.DataFrame.from_dict(v_means, orient='index', columns=['Vmean']).to_csv('../output/VPmean.csv')
